## Check Red Pitaya OS, SSH, and SCPI service with function gen and acq

This notebook verifies:
- Reachability (`ping`)
- SSH login and OS info
- SCPI port (`5000`) availability
- Waveform genration and acqusition in all the channels


### Helper Funtions

In [1]:
import sys
import numpy as np
import redpitaya_scpi as scpi
import platform, shutil, socket, subprocess, paramiko, time

In [2]:
def ping_once(ip, timeout_s=1):
    ip = str(ip).strip()
    ping_bin = shutil.which("ping")
    if ping_bin is None:
        return False, "ping executable not found in PATH"

    system = platform.system().lower()
    if system == "windows": 
        cmd = [ping_bin, "-n", "1", "-w", str(int(timeout_s * 1000)), ip]
    elif system == "darwin":
        # macOS ping uses milliseconds for -W
        cmd = [ping_bin, "-c", "1", "-W", str(int(timeout_s * 1000)), ip]
    else:
        # Linux ping uses seconds for -W
        cmd = [ping_bin, "-c", "1", "-W", str(max(1, int(timeout_s))), ip]

    proc = subprocess.run(cmd, capture_output=True, text=True)
    ok = proc.returncode == 0
    details = (proc.stdout or proc.stderr).strip()
    return ok, details

In [3]:
def ensure_scpi_running(host, username="root", password="root"):

    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(host, username=username, password=password)

    # Check SCPI status
    stdin, stdout, stderr = ssh.exec_command("systemctl is-active redpitaya_scpi")
    scpi_status = stdout.read().decode().strip()

    # Check nginx status
    stdin, stdout, stderr = ssh.exec_command("systemctl is-active redpitaya_nginx")
    nginx_status = stdout.read().decode().strip()

    if scpi_status == "active" and nginx_status != "active":
        print("SCPI already running and nginx already stopped.")
        ssh.close()
        return

    # Stop nginx if needed
    if nginx_status == "active":
        ssh.exec_command("systemctl stop redpitaya_nginx")
        print("Stopped redpitaya_nginx.")

    # Start SCPI if needed
    if scpi_status != "active":
        ssh.exec_command("systemctl start redpitaya_scpi")
        print("Started redpitaya_scpi.")

    time.sleep(2)

    # Final verification
    stdin, stdout, stderr = ssh.exec_command("systemctl is-active redpitaya_scpi")
    status = stdout.read().decode().strip()

    ssh.close()

    if status != "active":
        raise RuntimeError("SCPI service not running")

    print("SCPI ready!")

### Testing Connection

In [4]:
# --- Hardware address configuration ---
RP_CAV_IP = "192.168.0.101"   # scanning cavity RP

RPs = [RP_CAV_IP]

RP_SSH_USER = "root"
RP_SSH_PASS = "root"
SCPI_PORT = 5000

In [5]:
# Ping test
for ip in RPs:
    ok, details = ping_once(ip)
    print(f"Ping {ip}:", "OK" if ok else "FAILED")
    if not ok:
        print("  ping output:", details[:300] if details else "(no output)")

Ping 192.168.0.101: OK


In [6]:
ensure_scpi_running(host=RP_CAV_IP)

SCPI already running and nginx already stopped.


### Testing waveform genration

In [7]:
import redpitaya_scpi as scpi
rp = scpi.scpi(RP_CAV_IP)

In [8]:
# Reset generator state and set OUT1
rp.tx_txt("GEN:RST")

# For Out 1
rp.tx_txt("SOUR1:FUNC TRIANGLE")      # chan=2, func="TRIANGLE"
rp.tx_txt("SOUR1:FREQ:FIX 200")       # frequency
rp.tx_txt("SOUR1:VOLT 0.5")           # amplitude 
rp.tx_txt("SOUR1:VOLT:OFFS 0.0")      # offset

# Enable o/p 1
rp.tx_txt("OUTPUT1:STATE ON")
rp.tx_txt("SOUR1:TRIG:INT")

In [9]:
# Set OUT2

# For Out 2
rp.tx_txt("SOUR2:FUNC SINE")      # chan=2, func="TRIANGLE"
rp.tx_txt("SOUR2:FREQ:FIX 200")       # freqquency
rp.tx_txt("SOUR2:VOLT 0.5")           # amplitude
rp.tx_txt("SOUR2:VOLT:OFFS 0.0")      # offset

# Enable o/p 2
rp.tx_txt("OUTPUT2:STATE ON")
rp.tx_txt("SOUR2:TRIG:INT")

In [12]:
rp.tx_txt("OUTPUT1:STATE OFF")
rp.tx_txt("OUTPUT2:STATE OFF")

### Acquisition not working currently!

In [14]:
import matplotlib.pyplot as plt

In [ ]:
# # Configure acquisition on IN1 (equivalent to acq_set parameters)
rp.tx_txt('ACQ:DEC 8')                # dec=8
rp.tx_txt('ACQ:TRIG:LEV 0')           # trig_lvl=0.0 (volts)
rp.tx_txt('ACQ:TRIG:LEV:UNIT VOLT')   # units="VOLTS"
rp.tx_txt('ACQ:DATA:FORMAT ASC')      # sample_format="ASCII"  
rp.tx_txt('ACQ:AVG ON')               # averaging=True
rp.tx_txt('ACQ:CH1:GAIN LV')          # gain=["LV","LV"] for CH1
rp.tx_txt('ACQ:CH2:GAIN LV')

rp.tx_txt("ACQ:RST")
rp.tx_txt("ACQ:START")
rp.tx_txt("ACQ:TRIG CH1_PE")

# Wait until trigger event arrives
timeout_s = 2.0
start = time.time()

while time.time() - start < timeout_s:
    status = rp.txrx_txt("ACQ:TRIG:STAT?")
    if status == "TD":
        break
    time.sleep(0.01)  # Smaller sleep for better responsiveness
else:
    raise TimeoutError("Acquisition trigger timeout. Check wiring/trigger settings.")

# Get acquisition data (manual parsing needed for ASCII format)
rp.tx_txt("ACQ:DATA?")  
raw_data = rp.txrx_txt("ACQ:DATA?").strip()
samples = [float(x) for x in raw_data.split(',')[:4096]]  # First 4096 samples


samples = np.array(samples, dtype=float)
print("Acquired samples:", samples.size, "min/max:", samples.min(), samples.max())


plt.figure(figsize=(10, 3))
plt.plot(samples, lw=1)
plt.title("IN1 acquisition while driving OUT2 triangular waveform")
plt.xlabel("Sample index")
plt.ylabel("Voltage [V]")
plt.grid(True, alpha=0.3)
plt.show()

In [13]:
rp.close()